In [1]:
import numpy as np
import pandas as pd
import joblib
import json

In [ ]:
kmeans = joblib.load("../artifacts/kmeans_model.pkl")

iso_vip = joblib.load("../artifacts/iso_vip.pkl")           
iso_regular = joblib.load("../artifacts/iso_reg.pkl")   
iso_occ = joblib.load("../artifacts/iso_occ.pkl")    
# Load model config (optional)
with open("../artifacts/model_config.json", "r") as f:
    model_config = json.load(f)

model_config

{'model_type': 'KMeans',
 'n_clusters': 4,
 'features': ['Recency', 'Frequency', 'Monetary_log', 'AvgOrderValue_log'],
 'preprocessing': ['log1p on Monetary & AvgOrderValue'],
 'silhouette_score': 0.5834400677277796}

In [14]:
CLUSTER_LABELS = {
    0: "At-Risk / Lost Customers",
    1: "Regular Customers",
    2: "Occasional Buyers",
    3: "VIP / High-Value Customers"
}

DECISION_RULES = {
    3: "Manual Review (High Risk)",        # VIP anomaly
    2: "Upsell / Growth Opportunity",      # Occasional anomaly
    1: "Monitor Behavior",                 # Regular anomaly
    0: "No Action (Ignore)"                # At-Risk
}


In [15]:
def validate_customer_input(recency, frequency, monetary):
    if any(v is None for v in [recency, frequency, monetary]):
        return False, "Missing input values"

    if not all(isinstance(v, (int, float)) for v in [recency, frequency, monetary]):
        return False, "Inputs must be numeric"

    if recency < 0 or frequency < 0 or monetary < 0:
        return False, "Negative values are not allowed"

    return True, "Valid input"

In [16]:
def preprocess_customer(recency, frequency, monetary):
    """
    Computes AvgOrderValue automatically and applies log transform
    """

    if frequency == 0:
        return None, "No Purchase / New Customer"

    avg_order_value = monetary / frequency

    X = np.array([[
        recency,
        frequency,
        np.log1p(monetary),
        np.log1p(avg_order_value)
    ]])

    return X, "OK"

In [ ]:
def predict_customer(recency, frequency, monetary):

    is_valid, message = validate_customer_input(
        recency, frequency, monetary
    )
    if not is_valid:
        return {
            "status": "error",
            "message": message
        }
    X, status_msg = preprocess_customer(recency, frequency, monetary)

    if X is None:
        return {
            "status": "ok",
            "segment": "No Purchase / New Customer",
            "anomaly": False,
            "recommended_action": "Ignore"
        }

    segment_id = int(kmeans.predict(X)[0])
    segment_label = CLUSTER_LABELS[segment_id]

    anomaly = False

    if segment_id == 3:      # VIP
        anomaly = iso_vip.predict(X)[0] == -1
    elif segment_id == 1:    # Regular
        anomaly = iso_regular.predict(X)[0] == -1
    elif segment_id == 2:    # Occasional
        anomaly = iso_occ.predict(X)[0] == -1

    action = DECISION_RULES[segment_id] if anomaly else "No Action"

    return {
        "status": "ok",
        "segment_id": segment_id,
        "segment": segment_label,
        "anomaly": anomaly,
        "recommended_action": action
    }

In [35]:
predict_customer(
    recency=0,
    frequency=0,
    monetary=10
)

{'status': 'ok',
 'segment': 'No Purchase / New Customer',
 'anomaly': False,
 'recommended_action': 'Ignore'}